In [1]:
from collections import Counter
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from data.datasets import get_dataset
from data.splits import get_train_val_split
from data.transforms import image_transform

dataset_name = "Oxford-IIIT Pet classification"
data_dir = repo_root / "data" / "raw" / "iit_pet"


print(f"reading {dataset_name} trainval dataset to {data_dir}...")
trainval_dataset = get_dataset(
    split="trainval",
    dataset=dataset_name,
    data_dir=data_dir,
    download=False,
    transform=image_transform
)
print(f"reading {dataset_name} test dataset to {data_dir}...")
test_dataset = get_dataset(
    split="test",
    dataset=dataset_name,
    data_dir=data_dir,
    download=False,
    transform=image_transform
)

raw_train_dataset, raw_val_dataset = get_train_val_split(
    trainval_dataset,
    ratio=0.2,
    seed=42,
    labels=trainval_dataset._labels,
)

for name, subset in (("Train", raw_train_dataset), ("Validation", raw_val_dataset)):
    counts = Counter(trainval_dataset._labels[i] for i in subset.indices)
    print(
        f"{name}: {len(subset)} images, {len(counts)} classes "
        f"({min(counts.values())}-{max(counts.values())} per class)"
    )

print(f"Test: {len(test_dataset)} images (official held-out set)")
image, label = raw_train_dataset[0]
print(f"Example: image size={image.size}, label={label}")


reading Oxford-IIIT Pet classification trainval dataset to /Users/kasrasadatsharifi/SideProjects/ConvNeXts/data/raw/iit_pet...
reading Oxford-IIIT Pet classification test dataset to /Users/kasrasadatsharifi/SideProjects/ConvNeXts/data/raw/iit_pet...
Train: 2944 images, 37 classes (74-80 per class)
Validation: 736 images, 37 classes (19-20 per class)
Test: 3669 images (official held-out set)
Example: image size=<built-in method size of Tensor object at 0x11d6d4500>, label=33


In [2]:
from data.dataloaders import get_dataloader

NUM_WORKERS = 4
BATCH_SIZE = 32

train_dataloader = get_dataloader(
    dataset=raw_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_dataloader = get_dataloader(
    dataset=raw_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_dataloader = get_dataloader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

In [3]:
from models import get_classification_model

convnext_model = get_classification_model("convnext_tiny", pretrained=True, num_classes=37)
swin_transformer_model = get_classification_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=37)
resnet50_model = get_classification_model("resnet50", pretrained=True, num_classes=37)

/Users/kasrasadatsharifi/Library/Caches/pypoetry/virtualenvs/convnexts-2-MddwQV-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# fine tune the classifier heads

In [4]:
import torch
import tqdm
from evaluation import *

NUM_EPOCHS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
CRITERION = torch.nn.CrossEntropyLoss()

for model_name, model in (("ConvNeXt Tiny", convnext_model), ("Swin Transformer Tiny", swin_transformer_model), ("ResNet-50", resnet50_model)):
    print(f"fine tuning {model_name} for {NUM_EPOCHS} epochs...")
    model.to(DEVICE)
    OPTIMIZER = torch.optim.AdamW(model.parameters(), lr=1e-4)
    for i in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_dataloader, CRITERION, OPTIMIZER, DEVICE)
        val_loss, val_acc = validate(model, val_dataloader, CRITERION, DEVICE)
        print(f"Epoch {i+1}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

fine tuning ConvNeXt Tiny for 10 epochs...


Fine tuning ConvNeXt Tiny:  10%|█         | 1/10 [01:22<12:22, 82.51s/it]

Epoch 1/10: train_loss=0.8594, train_acc=0.7639, val_loss=0.3625, val_acc=0.8845


Fine tuning ConvNeXt Tiny:  20%|██        | 2/10 [02:44<10:59, 82.38s/it]

Epoch 2/10: train_loss=0.1654, train_acc=0.9484, val_loss=0.3603, val_acc=0.8967


...

Epoch 8/10: train_loss=0.0861, train_acc=0.9922, val_loss=0.2031, val_acc=0.9416


Fine tuning ResNet-50:  90%|█████████ | 9/10 [11:15<01:13, 73.46s/it]

Epoch 9/10: train_loss=0.0681, train_acc=0.9935, val_loss=0.2051, val_acc=0.9375


Fine tuning ResNet-50: 100%|██████████| 10/10 [12:27<00:00, 74.77s/it]

Epoch 10/10: train_loss=0.0545, train_acc=0.9949, val_loss=0.1991, val_acc=0.9348


# fine tune more layers